## 衆議院議員収集

In [11]:
import bs4
import requests
import re
from urllib.parse import urljoin
import os
from params.paths import ROOT_DIR
import pandas as pd
import time
from tqdm import tqdm
from file_handling.file_read_writer import write_json, read_json

SHUGIIN_REPR_URL = 'https://kokkai.sugawarataku.net/giin/rgiin.html'
LOWER_HOUSE_DATA_DIR = os.path.join(ROOT_DIR, 'data', 'data_shugiin', 'repr_list')
LOWER_HOUSE_DATA_HISTORICAL_TMP = os.path.join(ROOT_DIR, 'data', 'data_shugiin', 'repr_list', 'historical')
os.makedirs(LOWER_HOUSE_DATA_DIR, exist_ok=True)

In [37]:
class ShugiinCollector:
	def __init__(self):
		pass

	def process_one_election_row(self, row):
		year = row.find('div', {'class': 'el1'}).text
		month = row.find('div', {'class': 'el2'}).text
		day = row.find('div', {'class': 'el3'}).text
		election_name = row.find('div', {'class': 'el4'}).text
		district = row.find('div', {'class': 'el5'}).text
		party = row.find('div', {'class': 'el6'}).text
		result = row.find('div', {'class': 'el7'}).text
		election_freq = row.find('div', {'class': 'el8'}).text
		return {'year': year, 'month': month, 'day': day, 'election_name': election_name, 'district': district, 'party': party, 'result': result, 'election_freq': election_freq}

	def retrieve_info_of_one_repr(self, url):
		html = requests.get(url).content
		soup = bs4.BeautifulSoup(html, 'html.parser')
		repr_data = soup.find_all('div', {'class':'jt2'})
		name_kanji = repr_data[0].text
		name_kana = repr_data[1].text
		years = re.findall(r"\d{4}/\d{2}/\d{2}", repr_data[3].text)
		years = [year.replace('/', '-') for year in years]

		election_data = soup.find_all('div', {'class':'em1'})
		election_data = [self.process_one_election_row(row) for row in election_data]

		return {'name_kanji': name_kanji, 'name_kana': name_kana, 'years': years, 'election_data': election_data}

	def collect(self):
		base_html = requests.get(SHUGIIN_REPR_URL).content
		soup = bs4.BeautifulSoup(base_html, 'html.parser')
		links = soup.find_all('span', {'class':'zt5'})
		names = [link.find('a').text for link in links]
		hrefs = [link.find('a').get('href') for link in links]

		for idx, (name, href) in enumerate(zip(names, hrefs)):
			repr_path = os.path.join(LOWER_HOUSE_DATA_HISTORICAL_TMP, f'{name}.json')
			if os.path.exists(repr_path):
				print(f'{name} already exists')
				repr_data = read_json(repr_path)
				continue
			print(f'Processing {name}-{idx/len(names)*100:.2f}%')
			repr_link = urljoin(SHUGIIN_REPR_URL, href)
			repr_data = self.retrieve_info_of_one_repr(repr_link)
			write_json(repr_data, repr_path)
			time.sleep(1)

		all_repr_data = []
		for name in names:
			repr_path = os.path.join(LOWER_HOUSE_DATA_HISTORICAL_TMP, f'{name}.json')
			repr_data = read_json(repr_path)
			all_repr_data.append(repr_data)

		write_json({'data':all_repr_data}, os.path.join(LOWER_HOUSE_DATA_DIR, 'historical.json'))

In [38]:
sc = ShugiinCollector()
sc.collect()


相川勝六 already exists
逢沢一郎 already exists
逢沢寛 already exists
合沢栄 already exists
相沢武彦 already exists
逢沢英雄 already exists
相沢英之 already exists
愛知和男 already exists
愛知揆一 already exists
愛野興一郎 already exists
愛野時一郎 already exists
相原しの already exists
青木愛 already exists
青木清左衛門 already exists
青木孝義 already exists
青木宏之 already exists
青木正 already exists
青木正久 already exists
青野武一 already exists
青柳一郎 already exists
青柳高一 already exists
青柳仁士 already exists
青柳盛雄 already exists
青柳陽一郎 already exists
青山周平 already exists
青山丘 already exists
青山二三 already exists
青山雅幸 already exists
青山大人 already exists
赤池誠章 already exists
赤枝恒雄 already exists
赤城徳彦 already exists
赤木正幸 already exists
赤城宗徳 already exists
赤沢正道 already exists
赤沢亮正 already exists
赤路友蔵 already exists
茜ケ久保重光 already exists
赤羽一嘉 already exists
赤間二郎 already exists
赤松勇 already exists
赤松広隆 already exists
赤松正雄 already exists
赤松明勅 already exists
赤嶺政賢 already exists
秋田大助 already exists
秋葉賢也 already exists
秋葉忠利 already exists
秋元司 already exists
秋本真利 already exists


## 参議院議員収集

In [7]:
from tabula import read_pdf

UPPER_HOUSE_DATA_DIR = os.path.join(ROOT_DIR, 'data', 'data_sangiin', 'repr_list')
MEMBER_LIST_PDF_PATH = os.path.join(ROOT_DIR, 'data', 'data_sangiin', '2022giin_list_a.pdf')

In [34]:
def gengo2seireki(gengo):

	gengo_num = int(gengo[1:])
	if '昭' in gengo:
		gengo_num += 1925
	elif '平' in gengo:
		gengo_num += 1988
	elif '令' in gengo:
		gengo_num += 2018
	return gengo_num

def extract_gengo(string):
	# Regex pattern: Match characters inside parentheses but NOT if followed by a date
    pattern = re.compile(r'\((昭\d{1,2}|平\d{1,2})(?!\.\d)\)')
    
    # Find all matches
    matches = pattern.findall(string)
    
    return matches

def apply_gengo2seireki(string):
	string = string.replace('元', '1')
	gengos = extract_gengo(string)
	seirekis = []
	for gengo in gengos:
		seireki = gengo2seireki(gengo)
		seirekis.append(str(seireki))
	return '-'.join(seirekis)

In [35]:
df = read_pdf(MEMBER_LIST_PDF_PATH, pages='all')

master_df = df[0]
for i in range(1, len(df)):
	master_df = master_df.append(df[i])
master_df = master_df.replace('\r', '', regex=True)
master_df = master_df.replace('\n', '', regex=True)
df_path = os.path.join(UPPER_HOUSE_DATA_DIR, '2022.csv')

master_df.to_csv(df_path, index=False)


Got stderr: Feb 18, 2025 7:51:08 PM org.apache.fontbox.ttf.CmapSubtable processSubtype14

/tmp/ipykernel_7546/3209786212.py:5: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  master_df = master_df.append(df[i])
/tmp/ipykernel_7546/3209786212.py:5: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  master_df = master_df.append(df[i])
/tmp/ipykernel_7546/3209786212.py:5: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  master_df = master_df.append(df[i])
/tmp/ipykernel_7546/3209786212.py:5: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  master_df = master_df.append(df[i])
/tmp/ipykernel_7546/3209786212.py:5: FutureWarning: The frame.append method is

In [ ]:
df = pd.read_csv(df_path)
df['西暦'] = df['選挙回次等'].apply(apply_gengo2seireki)


df.to_csv(os.path.join(UPPER_HOUSE_DATA_DIR, '2022_processed.csv'), index=False)
df.head()

,議員氏名,読み方,議員氏名(本名),性別,会派(最終),選挙回次等,選挙区(最終),備考,西暦
0,鮎川 金次郎,あいかわ きんじろう,NaN,男,自由民主党,第5回(昭34)、辞職(昭34.12.29),東京都,NaN,1959
1,鮎川 義介,あいかわ よしすけ,NaN,男,第十七控室,第3回(昭28)、第5回(昭34)、辞職(昭34.12.29),全国,NaN,1953-1959
2,相澤 重明,あいざわ しげあき,NaN,男,各派に属しない議員,第4回(昭31)、第6回(昭37),神奈川県,NaN,1956-1962
3,相沢 武彦,あいざわ たけひこ,NaN,男,公明党,第10回(昭49),北海道,NaN,1974
4,会田 長栄,あいた ちょうえい,NaN,男,日本社会党・護憲連合,第15回(平元),福島県,NaN,1989
